In [5]:
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import os
import glob

In [6]:
from sklearn.ensemble import VotingClassifier, StackingClassifier

folder_path = "secondStagepreprocess/balancedsecondstage"
csv_files = glob.glob(folder_path + "/*.csv")  # Adjust for different file types
file_names = [os.path.basename(file) for file in csv_files]

print(csv_files[0])
mydata=0
outputKNNList=[]
outputGNBList=[]
outputSVCList=[]
outputlRList=[]
outputDTList=[]
outputGBCList=[]
outputRFList=[]
voting_accuracy=[]
stackingACC=[]
AccDataset={}
algAccOut={}

for i in range(len(file_names)):
    mydata = pd.read_csv(folder_path + "/" + file_names[i])
    print(f"{file_names[i]} => shape: {mydata.shape}, nulls: {mydata.isnull().sum().sum()}")
    
    if mydata.isnull().values.any() or len(mydata.columns) < 3:
        print(f"⛔ Skipping {file_names[i]} due to nulls or insufficient columns.")
        continue

    x = mydata.iloc[:, 1:-1].to_numpy()
    y = mydata.iloc[:, -1].to_numpy()

    if len(np.unique(y)) < 2:
        print(f"⛔ Skipping {file_names[i]} due to only one class in target.")
        continue

    # Continue with fitting classifiers...

    
    # Create KNN classifier
    knn = KNeighborsClassifier(n_neighbors = 5)
    KNNscores=cross_val_score(knn,x,y,cv=10,scoring='f1' )
    
    #printing avg score
    avgKNNscores=round(np.mean(KNNscores),4)*100
    outputKNNList.append(avgKNNscores)
   # print("Knn Avg scores",avgKNNscores)
    #Create GaussianNB classifier
    gnb = GaussianNB()
    gnbscores=cross_val_score(gnb,x,y,cv=10,scoring='f1' )
    #print("GNB scores",gnbscores)
    avgGNBscores=round(np.mean(gnbscores),4)*100
    outputGNBList.append(avgGNBscores)
    #print("GNB Avg scores",avgGNBscores)
    #create SVC(support vector classifier
    Svc=SVC(probability=True)
    SVCscores=cross_val_score(Svc,x,y,cv=10,scoring='f1')
    avgSVCscores=round(np.mean(SVCscores),4)*100
    outputSVCList.append(avgSVCscores)
    
    #logistic Regresion
    # instantiate & fit
    lr=LogisticRegression(max_iter=1000)
    LrScore=cross_val_score(lr,x,y,cv=10,scoring='f1' )
    avglrScore=round(np.mean(LrScore),4)*100
    #print('lrscore',avglrScore)
    outputlRList.append(avglrScore)
    
    #Desion tree
    DT = DecisionTreeClassifier(min_samples_split=10,max_depth=3)
    DTScore=cross_val_score(DT,x,y,cv=10, scoring='f1')
    avgDTScore=round(np.mean(DTScore),4)*100
    #print('avgDtscore',avgDTScore)
    outputDTList.append(avgDTScore)

    #Random forest
# instantiate & fit
    RF = RandomForestClassifier(n_estimators=300,max_depth=3)
    RFScore=cross_val_score(RF,x,y,cv=10 ,scoring='f1')
    avgRFScore=round(np.mean(RFScore),4)*100
    #print('avgRFScore',avgRFScore)
    outputRFList.append(avgRFScore)
    
    #Gradient boosting
    gbc = GradientBoostingClassifier(n_estimators=100)
    GBCScore=cross_val_score(gbc,x,y,cv=10 ,scoring='f1')
    avgGBCScore=round(np.mean(GBCScore),4)*100
    #print('avgGBCScore',avgGBCScore)
    outputGBCList.append(avgGBCScore)
    
    #Cross Val byvoting 
    voting = VotingClassifier(estimators=[
        ('KNN', knn), ('Gnb', gnb),('svc',Svc),('lr',lr),('dt',DT)
        ,('rf',RF),('GBC',gbc)
    ], voting='hard',weights=[1,1,1,1,1,1,10])
    
    scores = cross_val_score(voting, x, y, cv=10)
    voting_accuracy.append(round(scores.mean(),4)*100)
    print("Cross-validated accuracy:", scores.mean())
    
#stacking ensemble classifier 
    # Define meta-model
    meta_model = LogisticRegression(max_iter=1000)
    # Build stacking model
    stacking_clf = StackingClassifier(
    estimators=[
        ('KNN', knn), ('Gnb', gnb),('svc',Svc),('lr',lr),('dt',DT)
        ,('rf',RF),('GBC',gbc)],
    final_estimator=meta_model,
    passthrough=True,
    cv=10  # cross-validation inside stacking
    )

# تدريب أو cross-validation:
    scores = cross_val_score(stacking_clf, x, y, cv=10, scoring='accuracy')
    stackingACC.append(round(scores.mean(), 4)*100)
    print("StackingClassifier avg accuracy:", round(scores.mean(), 3))


secondStagepreprocess/balancedsecondstage\complexitymetaDataset.csv
complexitymetaDataset.csv => shape: (176, 5), nulls: 0


KeyboardInterrupt: 

In [ ]:
AccDataset = {}
inner_keys = ['KNN', 'GNB', 'SVC', 'LR', 'DT', 'RF','GBC','voting', 'stacking'
               ]
values = [outputKNNList,outputGNBList,outputSVCList,outputlRList,outputDTList,outputRFList,outputGBCList,voting_accuracy,stackingACC
           ]
# تأكد إن عدد الملفات = عدد النتائج في القوائم
print(len(file_names))            # كم ملف عندك
print(len(outputKNNList))         # طول كل قائمة دقة
  
for idx, outer_key in enumerate(file_names):
    AccDataset[outer_key] = {}
    for i in range(len(inner_keys)):
        AccDataset[outer_key][inner_keys[i]] = values[i][idx]
print(AccDataset)

6
6
{'complexitymetaDataset.csv': {'KNN': np.float64(77.64999999999999), 'GNB': np.float64(65.16), 'SVC': np.float64(72.34), 'LR': np.float64(64.05), 'DT': np.float64(73.19), 'RF': np.float64(75.13), 'GBC': np.float64(88.35), 'voting': np.float64(86.96000000000001), 'stacking': np.float64(89.18)}, 'GeneralMetaDatset.csv': {'KNN': np.float64(79.38), 'GNB': np.float64(64.29), 'SVC': np.float64(19.55), 'LR': np.float64(44.1), 'DT': np.float64(71.19), 'RF': np.float64(79.28), 'GBC': np.float64(89.53999999999999), 'voting': np.float64(88.89), 'stacking': np.float64(89.44)}, 'infotheoryMetaDataset.csv': {'KNN': np.float64(80.61), 'GNB': np.float64(67.99), 'SVC': np.float64(73.18), 'LR': np.float64(70.89999999999999), 'DT': np.float64(68.39), 'RF': np.float64(78.69), 'GBC': np.float64(92.63), 'voting': np.float64(91.67), 'stacking': np.float64(93.89)}, 'landmarkMeta.csv': {'KNN': np.float64(79.46), 'GNB': np.float64(75.58), 'SVC': np.float64(79.60000000000001), 'LR': np.float64(79.14), 'DT': 

In [ ]:
#['LR', 'SVC', 'GBC', 'GBC', 'GBC', 'GNB', 'LR', 'LR', 'KNN', 'SVC', 'KNN', 'RF']
maxAlgNames=[]
maxAlgAcc=[]
for dataName,algName in AccDataset.items():
    print(dataName,algName)
    for nameMaxalg in AccDataset[dataName]:
         print(max(AccDataset[dataName].values()))
         if AccDataset[dataName][nameMaxalg] == max(AccDataset[dataName].values()): 
             maxAlgNames.append(nameMaxalg)
             maxAlgAcc.append(max(AccDataset[dataName].values()))
             break
            
         
           
            
print(maxAlgNames)
print(maxAlgAcc)


    #saving  output in file
data = {
    'DatasetName': file_names,
    'KNN': outputKNNList,
    'GNB':outputGNBList,
    'SVC':outputSVCList,
    'LR':outputlRList,
    'DT':outputDTList,
    'RF':outputRFList,
    'GBC':outputGBCList,
    'ensemblevoting':voting_accuracy,
    'stacking':stackingACC,
    'max Acc':maxAlgAcc,
    'maxtechNames':maxAlgNames,
    
    
}
print(len(outputKNNList))
print(len(outputGNBList))
print(len(outputSVCList))
print(len(outputlRList))
print(len(outputDTList))
print(len(outputRFList))
print(len(outputGBCList))
print(len(maxAlgAcc))
print(len(maxAlgNames))
out_data=pd.DataFrame(data)
print(out_data)
file_name = 'outputs/secondStageOutData7metalearnerwith voting vs stackingf1.csv'
out_data.to_csv(file_name)
print('DataFrame is written to Excel File successfully.')

    

complexitymetaDataset.csv {'KNN': np.float64(77.64999999999999), 'GNB': np.float64(65.16), 'SVC': np.float64(72.34), 'LR': np.float64(64.05), 'DT': np.float64(73.19), 'RF': np.float64(75.13), 'GBC': np.float64(88.35), 'voting': np.float64(86.96000000000001), 'stacking': np.float64(89.18)}
89.18
89.18
89.18
89.18
89.18
89.18
89.18
89.18
89.18
GeneralMetaDatset.csv {'KNN': np.float64(79.38), 'GNB': np.float64(64.29), 'SVC': np.float64(19.55), 'LR': np.float64(44.1), 'DT': np.float64(71.19), 'RF': np.float64(79.28), 'GBC': np.float64(89.53999999999999), 'voting': np.float64(88.89), 'stacking': np.float64(89.44)}
89.53999999999999
89.53999999999999
89.53999999999999
89.53999999999999
89.53999999999999
89.53999999999999
89.53999999999999
infotheoryMetaDataset.csv {'KNN': np.float64(80.61), 'GNB': np.float64(67.99), 'SVC': np.float64(73.18), 'LR': np.float64(70.89999999999999), 'DT': np.float64(68.39), 'RF': np.float64(78.69), 'GBC': np.float64(92.63), 'voting': np.float64(91.67), 'stacking'